# 6. Hyperparameteroptimierung mit Optuna

In diesem Notebook werden die aussichtsreichsten klassischen Modelle aus der Baseline-Phase mit Optuna optimiert. Die Optimierung erfolgt ausschließlich auf dem Trainingsdatensatz. Der Testdatensatz bleibt weiterhin unberührt und wird erst in der finalen Evaluation verwendet.

## 6.1. Ziel der Optimierung

Die bisherigen Experimente zeigen, dass das Szenario `Text + categorical` eine gute Balance zwischen Modellleistung und reduzierter Merkmalskomplexität liefert. Daher wird dieses Szenario für die Hyperparameteroptimierung als feste Datengrundlage verwendet.

Die Reihenfolge der Optimierung orientiert sich an der bisherigen Laufzeit: Zuerst wird `SGDClassifier(loss="hinge")` optimiert, da dieses Modell sehr schnell trainiert und dadurch mehr Suchläufe erlaubt. Anschließend wird `Logistic Regression balanced` optimiert, da dieses Modell in der Baseline den höchsten Macro-F1-Score erzielt hat.

Als Hauptmetrik wird weiterhin `Macro-F1` verwendet. Zusätzlich werden Standardabweichung, Train-Test-Gap, Weighted-F1, Balanced Accuracy und Laufzeit dokumentiert, damit die Modellauswahl nicht nur auf einem einzelnen Mittelwert basiert.

In [ ]:
from pathlib import Path
import re
import tempfile

import mlflow
import numpy as np
import optuna
import pandas as pd

from IPython.display import display
from joblib import Memory, dump
from tqdm.auto import tqdm

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import f1_score, make_scorer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

## 6.2. Daten laden

Es werden nur die Trainingsdaten geladen. Die Hyperparameteroptimierung darf den Testdatensatz nicht verwenden, damit die finale Modellbewertung unverzerrt bleibt.

In [ ]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sklearn_memory = Memory(
    location=project_root / "cache" / "sklearn",
    verbose=0,
)

model_output_dir = project_root / "models" / "optuna"
model_output_dir.mkdir(parents=True, exist_ok=True)


def safe_model_filename(name):
    """Erzeugt einen sicheren Dateinamen für gespeicherte Modelle."""
    safe_name = re.sub(r"[^0-9a-zA-Z_-]+", "_", str(name))
    return safe_name.strip("_").lower()


def save_fitted_model(model, model_name, X, y):
    """Fitten auf allen Trainingsdaten und als Joblib-Datei speichern."""
    model_path = model_output_dir / f"{safe_model_filename(model_name)}.joblib"
    model.fit(X, y)
    dump(model, model_path)
    return model_path

train_df = pd.read_csv(
    project_root / "data" / "processed" / "train_data.csv",
    sep=";",
    index_col=0,
    encoding="utf-8",
)

print("Trainingsdaten:", train_df.shape)
display(train_df.head())

## 6.3. Optimierungsszenario festlegen

Für das Tuning wird das Szenario `Text + categorical` verwendet. Numerische Merkmale werden in diesem Szenario bewusst nicht verwendet, da die Ablation darauf hindeutet, dass die Text- und kategorialen Merkmale für diese Aufgabe ausreichend stark sind.

In [ ]:
target_column = "politikbereich"

text_features_optuna = [
    "name_standardised",
    "geber_standardised",
    "anschrift_standardised",
    "zweck_standardised",
]

categorical_features_optuna = [
    "art_standardised",
    "jahr",
]

numeric_features_optuna = []

feature_columns_optuna = (
    text_features_optuna
    + categorical_features_optuna
    + numeric_features_optuna
)

X_train = train_df[feature_columns_optuna].copy()
y_train = train_df[target_column].copy()

print("Verwendete Eingabespalten:")
display(pd.DataFrame({"Trainingsspalte": feature_columns_optuna}))

print("Anzahl Klassen:", y_train.nunique())

## 6.4. Vorverarbeitung und Evaluationsrahmen

Die Vorverarbeitung entspricht der Baseline-Logik: Jede Textspalte wird separat mit TF-IDF verarbeitet, kategoriale Merkmale werden One-Hot-codiert. Für die Bewertung wird Stratified K-Fold Cross-Validation verwendet, damit die Klassenverteilung in den Folds möglichst stabil bleibt.

In [ ]:
def flatten_column(values):
    """Konvertiert eine einzelne Spalte in ein eindimensionales String-Array."""
    return np.asarray(values, dtype=object).ravel()


text_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="",
            ),
        ),
        (
            "flatten",
            FunctionTransformer(
                flatten_column,
                validate=False,
            ),
        ),
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                min_df=2,
                max_df=0.90,
                ngram_range=(1, 2),
                sublinear_tf=True,
                max_features=150_000,
                dtype=np.float32,
            ),
        ),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore"),
        ),
    ]
)


def build_preprocessor(
    text_features,
    categorical_features,
):
    transformers = [
        (
            f"tfidf_{column}",
            text_transformer,
            [column],
        )
        for column in text_features
    ]

    if categorical_features:
        transformers.append(
            (
                "categorical",
                categorical_transformer,
                categorical_features,
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )


preprocessor_optuna = build_preprocessor(
    text_features_optuna,
    categorical_features_optuna,
)

n_splits = 4
cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=42,
)

scoring = {
    "macro_f1": "f1_macro",
    "weighted_f1": "f1_weighted",
    "balanced_accuracy": "balanced_accuracy",
    "accuracy": "accuracy",
}

main_metric = "macro_f1"

## 6.5. Hilfsfunktionen für Optuna

Für jeden Trial wird eine Cross-Validation durchgeführt. Optuna maximiert den durchschnittlichen Test-Macro-F1-Score. Zusätzlich werden Train-Macro-F1, Generalization Gap, Standardabweichung und Laufzeit gespeichert, damit Overfitting und Stabilität beurteilt werden können.

In [ ]:
def summarize_cv_results(cv_results):
    summary = {
        "fit_time_total_seconds": round(cv_results["fit_time"].sum(), 2),
        "fit_time_mean_seconds": round(cv_results["fit_time"].mean(), 2),
        "score_time_mean_seconds": round(cv_results["score_time"].mean(), 2),
    }

    for metric_name in scoring.keys():
        test_scores = cv_results[f"test_{metric_name}"]
        summary[f"test_{metric_name}_mean"] = round(test_scores.mean(), 4)
        summary[f"test_{metric_name}_std"] = round(test_scores.std(), 4)
        summary[f"{metric_name}_mean"] = summary[f"test_{metric_name}_mean"]
        summary[f"{metric_name}_std"] = summary[f"test_{metric_name}_std"]

        train_key = f"train_{metric_name}"

        if train_key in cv_results:
            train_scores = cv_results[train_key]
            summary[f"train_{metric_name}_mean"] = round(train_scores.mean(), 4)
            summary[f"train_{metric_name}_std"] = round(train_scores.std(), 4)
            summary[f"generalization_gap_{metric_name}"] = round(
                summary[f"train_{metric_name}_mean"]
                - summary[f"test_{metric_name}_mean"],
                4,
            )

    return summary


def evaluate_model(model):
    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=True,
    )

    return summarize_cv_results(cv_results)


def log_optuna_trial(trial, model_name, summary):
    trial.set_user_attr("model_name", model_name)

    for metric_name, metric_value in summary.items():
        if pd.notna(metric_value):
            trial.set_user_attr(metric_name, metric_value)


def build_result_table(study, model_name):
    rows = []

    for trial in study.trials:
        if trial.value is None:
            continue

        row = {
            "Modell": model_name,
            "trial_number": trial.number,
            "objective_value": trial.value,
        }
        row.update(trial.params)
        row.update(trial.user_attrs)
        rows.append(row)

    return (
        pd.DataFrame(rows)
        .sort_values("objective_value", ascending=False)
        .reset_index(drop=True)
    )

## 6.6. Optuna und MLflow vorbereiten

Die Optuna-Studien werden in einer SQLite-Datei gespeichert. Dadurch können bereits berechnete Trials später wiederverwendet werden. Zusätzlich werden die besten Ergebnisse als CSV-Dateien gespeichert und in MLflow dokumentiert.

In [ ]:
optuna_storage_path = (
    project_root
    / "data"
    / "processed"
    / "optuna_studies.db"
)

optuna_storage_url = "sqlite:///" + optuna_storage_path.as_posix()

optuna_results_dir = project_root / "data" / "processed"
optuna_results_dir.mkdir(parents=True, exist_ok=True)

mlflow_tracking_uri = (project_root / "mlruns").as_uri()
mlflow.set_tracking_uri(mlflow_tracking_uri)
mlflow.set_experiment("politikbereich_classifier_optuna")

sampler = optuna.samplers.TPESampler(seed=42)

print("Optuna storage:", optuna_storage_url)
print("MLflow tracking URI:", mlflow_tracking_uri)

## 6.7. Studie 1: SGDClassifier mit Hinge Loss

Zuerst wird das schnelle SGD-SVM-Modell optimiert. Der Schwerpunkt liegt auf der Regularisierung (`alpha`), der Penalty-Struktur und der Konvergenztoleranz. `class_weight="balanced"` bleibt gesetzt, weil die Zielvariable deutlich unausgewogen ist.

In [ ]:
def objective_sgd_hinge(trial):
    penalty = trial.suggest_categorical(
        "penalty",
        [
            "l2",
            "elasticnet",
        ],
    )

    classifier_params = {
        "loss": "hinge",
        "penalty": penalty,
        "alpha": trial.suggest_float(
            "alpha",
            1e-6,
            1e-3,
            log=True,
        ),
        "class_weight": "balanced",
        "max_iter": trial.suggest_categorical(
            "max_iter",
            [
                1000,
                2000,
                5000,
            ],
        ),
        "tol": trial.suggest_categorical(
            "tol",
            [
                1e-4,
                1e-3,
                1e-2,
            ],
        ),
        "average": trial.suggest_categorical(
            "average",
            [
                False,
                True,
            ],
        ),
        "random_state": 42,
        "n_jobs": -1,
    }

    if penalty == "elasticnet":
        classifier_params["l1_ratio"] = trial.suggest_float(
            "l1_ratio",
            0.05,
            0.50,
        )

    model = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor_optuna,
            ),
            (
                "classifier",
                SGDClassifier(**classifier_params),
            ),
        ],
        memory=sklearn_memory,
    )

    summary = evaluate_model(model)
    log_optuna_trial(
        trial,
        "SGD hinge",
        summary,
    )

    return summary[f"test_{main_metric}_mean"]


sgd_hinge_study = optuna.create_study(
    study_name="sgd_hinge_text_categorical_macro_f1",
    direction="maximize",
    storage=optuna_storage_url,
    load_if_exists=True,
    sampler=sampler,
)

n_trials_sgd_hinge = 50

sgd_hinge_study.optimize(
    objective_sgd_hinge,
    n_trials=n_trials_sgd_hinge,
    show_progress_bar=True,
)

sgd_hinge_results = build_result_table(
    sgd_hinge_study,
    "SGD hinge",
)

display(sgd_hinge_results.head(10))

## 6.8. Studie 2: Logistische Regression

Anschließend wird die reguläre logistische Regression optimiert. Dieses Modell war in der Baseline sehr stark, ist aber rechenintensiver als das SGD-Modell. Deshalb wird es nach dem schnellen SGD-Modell optimiert.

In [ ]:
def objective_logistic_regression(trial):
    penalty = trial.suggest_categorical(
        "penalty",
        [
            "l2",
            "l1",
        ],
    )

    classifier_params = {
        "C": trial.suggest_float(
            "C",
            1e-2,
            10.0,
            log=True,
        ),
        "penalty": penalty,
        "solver": "saga",
        "class_weight": trial.suggest_categorical(
            "class_weight",
            [
                "balanced",
                None,
            ],
        ),
        "max_iter": trial.suggest_categorical(
            "max_iter",
            [
                1000,
                2000,
                5000,
            ],
        ),
        "tol": trial.suggest_categorical(
            "tol",
            [
                1e-4,
                1e-3,
                1e-2,
            ],
        ),
        "n_jobs": -1,
        "random_state": 42,
    }

    model = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor_optuna,
            ),
            (
                "classifier",
                LogisticRegression(**classifier_params),
            ),
        ],
        memory=sklearn_memory,
    )

    summary = evaluate_model(model)
    log_optuna_trial(
        trial,
        "Logistic Regression",
        summary,
    )

    return summary[f"test_{main_metric}_mean"]


logreg_study = optuna.create_study(
    study_name="logreg_text_categorical_macro_f1",
    direction="maximize",
    storage=optuna_storage_url,
    load_if_exists=True,
    sampler=sampler,
)

n_trials_logreg = 30

logreg_study.optimize(
    objective_logistic_regression,
    n_trials=n_trials_logreg,
    show_progress_bar=True,
)

logreg_results = build_result_table(
    logreg_study,
    "Logistic Regression",
)

display(logreg_results.head(10))

## 6.9. Ergebnisse vergleichen und speichern

Zum Abschluss werden die besten Trials beider Studien zusammengeführt. Für die finale Auswahl werden nicht nur Macro-F1, sondern auch Standardabweichung, Generalization Gap und Laufzeit betrachtet.

In [ ]:
optuna_results = pd.concat(
    [
        sgd_hinge_results,
        logreg_results,
    ],
    ignore_index=True,
)

optuna_results = (
    optuna_results
    .sort_values(
        "objective_value",
        ascending=False,
    )
    .reset_index(drop=True)
)

important_columns = [
    column
    for column in optuna_results.columns
    if column in [
        "Modell",
        "trial_number",
        "objective_value",
        "penalty",
        "alpha",
        "l1_ratio",
        "C",
        "class_weight",
        "max_iter",
        "tol",
        "average",
    ]
    or column.endswith("_mean")
    or column.endswith("_std")
    or column.startswith("generalization_gap_")
    or column.endswith("_seconds")
]

display(
    optuna_results[important_columns].head(20)
)

optuna_results_path = (
    optuna_results_dir
    / "optuna_tuning_results.csv"
)

sgd_hinge_results_path = (
    optuna_results_dir
    / "optuna_sgd_hinge_results.csv"
)

logreg_results_path = (
    optuna_results_dir
    / "optuna_logreg_results.csv"
)

optuna_results.to_csv(
    optuna_results_path,
    index=False,
    sep=";",
    encoding="utf-8",
)

sgd_hinge_results.to_csv(
    sgd_hinge_results_path,
    index=False,
    sep=";",
    encoding="utf-8",
)

logreg_results.to_csv(
    logreg_results_path,
    index=False,
    sep=";",
    encoding="utf-8",
)

print("Gespeichert:", optuna_results_path)
print("Gespeichert:", sgd_hinge_results_path)
print("Gespeichert:", logreg_results_path)

## 6.10. Bestes Modell für die finale Evaluation auswählen

Der beste Trial nach Macro-F1 ist ein erster Kandidat für die finale Evaluation. Vor der endgültigen Auswahl sollten jedoch auch `generalization_gap_macro_f1`, `macro_f1_std` und die Laufzeit betrachtet werden. Ein sehr hoher Train-Test-Gap kann auf Overfitting hinweisen.

In [ ]:
best_trial_summary = optuna_results.iloc[0]

print("Bestes Modell:", best_trial_summary["Modell"])
print("Trial:", best_trial_summary["trial_number"])
print("Macro-F1:", best_trial_summary["test_macro_f1_mean"])

if "generalization_gap_macro_f1" in best_trial_summary:
    print(
        "Generalization Gap Macro-F1:",
        best_trial_summary["generalization_gap_macro_f1"],
    )

display(
    best_trial_summary.to_frame("Wert")
)

best_params = {
    key: value
    for key, value in best_trial_summary.items()
    if key in [
        "penalty",
        "alpha",
        "l1_ratio",
        "C",
        "class_weight",
        "max_iter",
        "tol",
        "average",
    ]
    and pd.notna(value)
}

if best_trial_summary["Modell"] == "SGD hinge":
    classifier_params = {
        "loss": "hinge",
        "penalty": best_params.get("penalty", "l2"),
        "alpha": float(best_params.get("alpha", 1e-4)),
        "class_weight": "balanced",
        "max_iter": int(best_params.get("max_iter", 1000)),
        "tol": float(best_params.get("tol", 1e-3)),
        "average": bool(best_params.get("average", False)),
        "random_state": 42,
        "n_jobs": -1,
    }

    if classifier_params["penalty"] == "elasticnet":
        classifier_params["l1_ratio"] = float(
            best_params.get("l1_ratio", 0.15)
        )

    final_classifier = SGDClassifier(**classifier_params)

elif best_trial_summary["Modell"] == "Logistic Regression":
    final_classifier = LogisticRegression(
        C=float(best_params.get("C", 1.0)),
        penalty=best_params.get("penalty", "l2"),
        solver="saga",
        class_weight=best_params.get("class_weight", "balanced"),
        max_iter=int(best_params.get("max_iter", 2000)),
        tol=float(best_params.get("tol", 1e-3)),
        n_jobs=-1,
        random_state=42,
    )

else:
    raise ValueError(
        f"Unbekanntes Modell: {best_trial_summary['Modell']}"
    )

best_model_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor_optuna,
        ),
        (
            "classifier",
            final_classifier,
        ),
    ],
    memory=sklearn_memory,
)

best_model_path = save_fitted_model(
    best_model_pipeline,
    f"best_optuna_{best_trial_summary['Modell']}",
    X_train,
    y_train,
)

print("Gespeichertes finales Optuna-Modell:", best_model_path)
